In [3]:
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import gc
from catboost import CatBoostRegressor, Pool
from optuna.integration import CatBoostPruningCallback
import optuna
import numpy as np

/Users/filimono/Documents/Innowise/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/importlib/__init__.py:88: FutureWarning: `optuna.integration.catboost` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.catboost` instead.
  return _bootstrap._gcd_import(name[level:], package, level)


In [7]:
df = pd.read_parquet('../data/final_data/data.parquet')

In [4]:
test = pd.read_parquet('../data/final_data/test.parquet')

In [8]:
feature_cols = [c for c in df.columns
                if c not in [
                    "item_cnt_month",
                    "date_block_num",
                    "avg_price",
                    "item_avg_cnt",
                    "shop_avg_cnt",
                    "cat_avg_cnt",
                    "cat_month_avg",
                ]]

In [9]:
X_test = test[feature_cols]

In [10]:
extra = df[df["date_block_num"] == 33][
    ["shop_id", "item_id", "item_avg_cnt", "shop_avg_cnt", "item_category_id", "cat_avg_cnt"]
].drop_duplicates(subset=["shop_id", "item_id"])

test = test.merge(extra[["shop_id", "item_id", "item_avg_cnt", "shop_avg_cnt"]], 
                  on=["shop_id", "item_id"], how="left")
test = test.merge(extra[["item_category_id", "cat_avg_cnt"]].drop_duplicates("item_category_id"),
                  on="item_category_id", how="left")

In [11]:
item_lag1_global = (df[df.date_block_num == 33]
                    .groupby("item_id")["item_cnt_month"]
                    .sum().reset_index()
                    .rename(columns={"item_cnt_month": "item_lag1_global"}))

item_active = (df[df.date_block_num.isin([31,32,33])]
               .groupby("item_id")["item_cnt_month"]
               .apply(lambda x: (x > 0).sum()).reset_index()
               .rename(columns={"item_cnt_month": "item_active_months"}))


df = df.merge(item_lag1_global, on="item_id", how="left")
df = df.merge(item_active, on="item_id", how="left")


test = test.merge(item_lag1_global, on="item_id", how="left")
test = test.merge(item_active, on="item_id", how="left")

---

## После EA

In [12]:
item_stats = (df.groupby(["item_id", "date_block_num"])["item_cnt_month"]
                .mean().reset_index()
                .sort_values("date_block_num"))

In [13]:
item_stats["item_max_sales"] = (item_stats.groupby("item_id")["item_cnt_month"]
                                 .transform(lambda x: x.shift(1).expanding().max()))
item_stats["item_q90_sales"] = (item_stats.groupby("item_id")["item_cnt_month"]
                                 .transform(lambda x: x.shift(1).expanding().quantile(0.9)))

In [14]:
item_stats = item_stats[["item_id", "date_block_num", "item_max_sales", "item_q90_sales"]]

In [15]:
df = df.merge(item_stats, on=["item_id", "date_block_num"], how="left")

In [16]:
item_stats_test = item_stats[item_stats.date_block_num == 33][
    ["item_id", "item_max_sales", "item_q90_sales"]
]
test = test.merge(item_stats_test, on="item_id", how="left")

In [17]:
df["was_sold_last_3m"] = (
    df[["item_cnt_month_lag_1", "item_cnt_month_lag_2", "item_cnt_month_lag_3"]]
    .gt(0).any(axis=1).astype(int)
)
test["was_sold_last_3m"] = (
    test[["item_cnt_month_lag_1", "item_cnt_month_lag_2", "item_cnt_month_lag_3"]]
    .gt(0).any(axis=1).astype(int)
)

In [18]:
cat_month_stats = (df.groupby(["item_category_id", "date_block_num", "month"])
                     ["item_cnt_month"].mean().reset_index()
                     .sort_values("date_block_num"))

cat_month_stats["cat_month_avg"] = (
    cat_month_stats.groupby(["item_category_id", "month"])["item_cnt_month"]
    .transform(lambda x: x.shift(1).expanding().mean())
)
cat_month_stats = cat_month_stats[["item_category_id", "date_block_num", "cat_month_avg"]]

In [19]:
df = df.merge(cat_month_stats, on=["item_category_id", "date_block_num"], how="left")

In [20]:
cat_month_test = cat_month_stats[cat_month_stats.date_block_num == 33][
    ["item_category_id", "cat_month_avg"]
]

In [21]:
test = test.merge(cat_month_test, on="item_category_id", how="left")

In [22]:
cat_dyn = (df.groupby(["item_category_id", "date_block_num"])["item_cnt_month"]
             .mean().reset_index()
             .sort_values("date_block_num"))

cat_dyn["cat_trend_3m"] = (cat_dyn.groupby("item_category_id")["item_cnt_month"]
                            .transform(lambda x: x.shift(1).diff(3)))

cat_dyn = cat_dyn[["item_category_id", "date_block_num", "cat_trend_3m"]]

In [23]:
df = df.merge(cat_dyn, on=["item_category_id", "date_block_num"], how="left")

In [24]:
cat_trend_test = cat_dyn[cat_dyn.date_block_num == 33][
    ["item_category_id", "cat_trend_3m"]
]
test = test.merge(cat_trend_test, on="item_category_id", how="left")

In [25]:
cat_historical = (df[df.date_block_num < 32]
                  .groupby("item_category_id")["item_cnt_month"]
                  .mean().reset_index()
                  .rename(columns={"item_cnt_month": "cat_historical_mean"}))

cat_last = (df[df.date_block_num == 32]
            .groupby("item_category_id")["item_cnt_month"]
            .mean().reset_index()
            .rename(columns={"item_cnt_month": "cat_last_mean"}))

In [26]:
cat_ratio = cat_historical.merge(cat_last, on="item_category_id", how="left")
cat_ratio["cat_last_vs_mean"] = cat_ratio["cat_last_mean"] / (cat_ratio["cat_historical_mean"] + 1)
cat_ratio = cat_ratio[["item_category_id", "cat_last_vs_mean"]]

In [27]:
df   = df.merge(cat_ratio, on="item_category_id", how="left")
test = test.merge(cat_ratio, on="item_category_id", how="left")

In [28]:
print(test["month"].unique())

[10]


---

In [29]:
df['shop_id'] = df['shop_id'].astype('int8')
df['item_id'] = df['item_id'].astype('int16')
df['date_block_num'] = df['date_block_num'].astype('int8')   # от -128 до 127
df['month'] = df['month'].astype('int8')
df['year'] = df['year'].astype('int16')           
df['item_category_id'] = df['item_category_id'].astype('int8')
df['was_sold_last_3m'] = df['was_sold_last_3m'].astype('int8')
df['item_cnt_month'] = df['item_cnt_month'].astype('float16')
df['avg_price'] = df['avg_price'].astype('float16')

df['item_cnt_month_lag_1'] = df['item_cnt_month_lag_1'].astype('float32')
df['item_cnt_month_lag_2'] = df['item_cnt_month_lag_2'].astype('float32')
df['item_cnt_month_lag_3'] = df['item_cnt_month_lag_3'].astype('float32')
df['item_cnt_month_lag_6'] = df['item_cnt_month_lag_6'].astype('float32')
df['item_cnt_month_lag_12'] = df['item_cnt_month_lag_12'].astype('float32')

df['item_avg_cnt'] = df['item_avg_cnt'].astype('float32')
df['item_avg_cnt_lag_1'] = df['item_avg_cnt_lag_1'].astype('float32')
df['item_avg_cnt_lag_2'] = df['item_avg_cnt_lag_2'].astype('float32')
df['item_avg_cnt_lag_3'] = df['item_avg_cnt_lag_3'].astype('float32')

df['shop_avg_cnt'] = df['shop_avg_cnt'].astype('float32')
df['shop_avg_cnt_lag_1'] = df['shop_avg_cnt_lag_1'].astype('float32')
df['shop_avg_cnt_lag_2'] = df['shop_avg_cnt_lag_2'].astype('float32')
df['shop_avg_cnt_lag_3'] = df['shop_avg_cnt_lag_3'].astype('float32')

df['item_cnt_month_rmean_3'] = df['item_cnt_month_rmean_3'].astype('float32')
df['item_cnt_month_rmean_6'] = df['item_cnt_month_rmean_6'].astype('float32')
df['item_cnt_month_rmean_12'] = df['item_cnt_month_rmean_12'].astype('float32')

df['trend_1_2'] = df['trend_1_2'].astype('float32')
df['trend_1_12'] = df['trend_1_12'].astype('float32')


df['cat_avg_cnt'] = df['cat_avg_cnt'].astype('float32')
df['cat_avg_cnt_lag_1'] = df['cat_avg_cnt_lag_1'].astype('float32')

df['avg_price_lag_1'] = df['avg_price_lag_1'].astype('float32')
df['item_lag1_global'] = df['item_lag1_global'].astype('float32')

df['item_active_months'] = df['item_active_months'].astype('float32')
df['item_max_sales'] = df['item_max_sales'].astype('float32')
df['item_q90_sales'] = df['item_q90_sales'].astype('float32')

df['cat_month_avg'] = df['cat_month_avg'].astype('float32')
df['cat_trend_3m'] = df['cat_trend_3m'].astype('float32')
df['cat_last_vs_mean'] = df['cat_last_vs_mean'].astype('float32')


In [30]:
X_test  = test[feature_cols].copy()

In [31]:
test['shop_id'] = test['shop_id'].astype('int8')
test['item_id'] = test['item_id'].astype('int16')
test['date_block_num'] = test['date_block_num'].astype('int8')   # от -128 до 127
test['month'] = test['month'].astype('int8')
test['year'] = test['year'].astype('int16')           
test['item_category_id'] = test['item_category_id'].astype('int8')
test['was_sold_last_3m'] = test['was_sold_last_3m'].astype('int8')
test['item_cnt_month'] = test['item_cnt_month'].astype('float16')
test['avg_price'] = test['avg_price'].astype('float16')

test['item_cnt_month_lag_1'] = test['item_cnt_month_lag_1'].astype('float32')
test['item_cnt_month_lag_2'] = test['item_cnt_month_lag_2'].astype('float32')
test['item_cnt_month_lag_3'] = test['item_cnt_month_lag_3'].astype('float32')
test['item_cnt_month_lag_6'] = test['item_cnt_month_lag_6'].astype('float32')
test['item_cnt_month_lag_12'] = test['item_cnt_month_lag_12'].astype('float32')

test['item_avg_cnt'] = test['item_avg_cnt'].astype('float32')
test['item_avg_cnt_lag_1'] = test['item_avg_cnt_lag_1'].astype('float32')
test['item_avg_cnt_lag_2'] = test['item_avg_cnt_lag_2'].astype('float32')
test['item_avg_cnt_lag_3'] = test['item_avg_cnt_lag_3'].astype('float32')

test['shop_avg_cnt'] = test['shop_avg_cnt'].astype('float32')
test['shop_avg_cnt_lag_1'] = test['shop_avg_cnt_lag_1'].astype('float32')
test['shop_avg_cnt_lag_2'] = test['shop_avg_cnt_lag_2'].astype('float32')
test['shop_avg_cnt_lag_3'] = test['shop_avg_cnt_lag_3'].astype('float32')

test['item_cnt_month_rmean_3'] = test['item_cnt_month_rmean_3'].astype('float32')
test['item_cnt_month_rmean_6'] = test['item_cnt_month_rmean_6'].astype('float32')
test['item_cnt_month_rmean_12'] = test['item_cnt_month_rmean_12'].astype('float32')

test['trend_1_2'] = test['trend_1_2'].astype('float32')
test['trend_1_12'] = test['trend_1_12'].astype('float32')


test['cat_avg_cnt'] = test['cat_avg_cnt'].astype('float32')
test['cat_avg_cnt_lag_1'] = test['cat_avg_cnt_lag_1'].astype('float32')

test['avg_price_lag_1'] = test['avg_price_lag_1'].astype('float32')
test['item_lag1_global'] = test['item_lag1_global'].astype('float32')

test['item_active_months'] = test['item_active_months'].astype('float32')
test['item_max_sales'] = test['item_max_sales'].astype('float32')
test['item_q90_sales'] = test['item_q90_sales'].astype('float32')

test['cat_month_avg'] = test['cat_month_avg'].astype('float32')
test['cat_trend_3m'] = test['cat_trend_3m'].astype('float32')
test['cat_last_vs_mean'] = test['cat_last_vs_mean'].astype('float32')

In [32]:
drop = ['cat_last_vs_mean', 'cat_trend_3m', 'cat_month_avg', 'year', 'item_cnt_month_rmean_12', 'item_cnt_month_lag_12', 'item_active_months', 'item_lag1_global', 'trend_1_2', 'cat_avg_cnt_lag_1', 'cat_avg_cnt']

In [33]:
df.drop(drop, axis=1, inplace=True)

In [34]:
test.drop(drop, axis=1, inplace=True)

In [35]:
feature_cols = [c for c in df.columns
                if c not in [
                    "item_cnt_month",
                    "date_block_num",
                    "avg_price",
                    "item_avg_cnt",
                    "shop_avg_cnt",
                    "cat_avg_cnt",
                    "cat_month_avg",
                ]]

In [36]:
X_test  = test[feature_cols].copy()

---

## catboost

---

In [37]:
for col in ["shop_id", "item_id", "item_category_id"]:
    X_test[col]  = X_test[col].astype(str)

In [38]:
cat_features = ["shop_id", "item_id", "item_category_id"]

In [39]:
final_model = CatBoostRegressor()

model_path = '../Data/model/cat_model.cbm'
final_model.load_model(model_path)

CatBoostRegressor(bootstrap_type='Bernoulli', depth=6, devices='0', eval_metric='RMSE', iterations=350, l2_leaf_reg=0.5239386707, learning_rate=0.1378311523, loss_function='RMSE', min_data_in_leaf=144, random_seed=11, subsample=0.9969450662, use_best_model=False, verbose=100)

In [36]:
import json
with open("../Data/model/best_params.json", "r") as f:
  best_params = json.load(f)

In [37]:
final_params = best_params.copy()
final_params.update({
    "iterations": int(350),
    "random_seed": 11,
    "bootstrap_type": "Bernoulli",
    "eval_metric": "RMSE",
    "loss_function": "RMSE",
    "use_best_model": False,
    # "task_type": "GPU",
    "devices": "0",
    "verbose": 100,
})

In [38]:
final_params.pop("early_stopping_rounds", None)

In [39]:
X_train_final = df[(df.date_block_num >= 6) & (df.date_block_num <= 33)][feature_cols].copy()
y_train_final = df[(df.date_block_num >= 6) & (df.date_block_num <= 33)]["item_cnt_month"]

In [40]:
for col in cat_features:
    X_train_final[col] = X_train_final[col].astype(str)

In [41]:
final_model = CatBoostRegressor(**final_params)
final_model.fit(
    X_train_final,
    y_train_final,
    cat_features=cat_features
)

0:	learn: 1.1656627	total: 1.63s	remaining: 9m 27s
100:	learn: 0.7466357	total: 2m 35s	remaining: 6m 22s
200:	learn: 0.7175440	total: 6m 41s	remaining: 4m 57s
300:	learn: 0.6964916	total: 10m 35s	remaining: 1m 43s
349:	learn: 0.6878181	total: 12m 32s	remaining: 0us


CatBoostRegressor(bootstrap_type='Bernoulli', depth=6, devices='0', eval_metric='RMSE', iterations=350, l2_leaf_reg=0.5239386706825182, learning_rate=0.13783115227486023, loss_function='RMSE', min_data_in_leaf=144, random_seed=11, subsample=0.9969450661841349, use_best_model=False, verbose=100)

In [ ]:
# final_model.save_model('../Data/model/cat_model.cbm')

In [43]:
pred_cat = final_model.predict(X_test).clip(0, 20)

---

# WANDB

In [40]:
import sys
print("Python:", sys.executable)
%pip install -e .. -q

import wandb
import sales_ds
import subprocess
from catboost import CatBoostRegressor

run = wandb.init(
    project="predict-future-sales",
    name="catboost-final",
)

wandb.config.update({"ds_package_version": sales_ds.__version__})

try:
    dvc_commit = subprocess.check_output(
        ["git", "log", "--format=%H", "-n", "1", "Data/raw_data.dvc"],
        stderr=subprocess.DEVNULL
    ).decode().strip()
except Exception:
    dvc_commit = "not tracked"
wandb.config.update({"data_dvc_commit": dvc_commit})

artifact = wandb.Artifact("catboost-model", type="model")
artifact.add_file("../Data/model/cat_model.cbm")
run.log_artifact(artifact)

import pandas as pd
pred_cat = final_model.predict(X_test).clip(0, 20)
pd.DataFrame({
    "ID": test["ID"],
    "item_cnt_month": pred_cat
}).to_csv("../results/submission.csv", index=False)

wandb.finish()
print("Модель и предсказания залогированы в wandb")

Python: /Users/filimono/Documents/Innowise/.venv/bin/python

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: WARNING Artifact "catboost-model" already exists with the same content. No new version will be created.


Модель и предсказания залогированы в wandb


---